<img src="http://www.cidaen.es/assets/img/mCIDaeNnb.png" alt="Logo CiDAEN" align="right">




<br><br><br>
<h2><font color="#00586D" size=4>Módulo 8</font></h2>



<h1><font color="#00586D" size=5>Sistema de recomendación de libros</font></h1>

<br><br><br>
<div style="text-align: right">
<font color="#00586D" size=3>Luis de la Ossa</font><br>
<font color="#00586D" size=3>Máster en Ciencia de Datos e Ingeniería de Datos en la Nube</font><br>
<font color="#00586D" size=3>Universidad de Castilla-La Mancha</font>

</div>

---

<a id="indice"></a>
<h2><font color="#00586D" size=5>Índice</font></h2>


* [1. Introducción](#section1)
   * [Lectura de las revisiones](#section11)
   * [Etiquetas](#section12)
   * [Preparación de los datos](#section13)
* [2. Limpieza del texto](#section2)
* [3. Búsqueda por similaridad de la representación tf-idf](#section3)
* [4. Recomendación basada en contenido](#section4)
   * [Selección de libros preferidos por el usuario](#section41)
   * [Búsqueda de libros similares](#section42)
   * [Priorización de resultados y recomendación](#section43)
* [5. Sistema híbrido de recomendación](#section5)  
* [6. LDA y búsqueda por similaridad](#section6)
* [Anexo. Obtención de los resúmenes mediante scraping](#Anexo)
<br>

In [1]:
# Permite establecer la anchura de la celda
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
# Algunas inconsistencias con las versiones dan lugar a avisos molestos. Se ignoran.
import warnings
warnings.filterwarnings('ignore')

C:\Users\LUIS\AppData\Local\Temp\ipykernel_1336\2416196964.py:2: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


<a id="section1"></a>
## <font color="#00586D"> 1. Introducción</font>
<br>

Los sistemas de recomendación basados en filtrado colaborativo utilizan de manera exclusiva los perfiles de votación de usuarios/items, y no consideran el contenido. Como se vió en clase,  permiten obtener resultados que, si bien cualitativamente son aceptables, pueden llegar a desconcertar al no corresponderse con lo esperado. 

En este proyecto se diseñará un pequeño sistema de recomendación de libros basado en contenido. Para ello, será necesario hacer uso de algunos de los conceptos relaccionados con aprendizaje automático sobre información textual (_Text Mining_) que han sido tratados también a lo largo del módulo. 

Como punto de partida, se partirá del conjunto de datos [goodbooks-10k](https://www.kaggle.com/zygmunt/goodbooks-10k) disponible en [kaggle](https://www.kaggle.com). Éste contiene información relativa a 10000 libros obtenida de la red social  [goodreads](http://goodreads.com), que actualmente es el sitio de referencia en la red para aficcionados a la lectura. Además de títulos y autores, el conjunto de datos incluye votos y etiquetas aportadas por más de 53000 usuarios. 

<div class="alert alert-block alert-warning">
<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
Se han hecho algunas modificaciones con respecto a la base original para que sea menos tedioso manejar los distintos índices e identificadores. 
</div>

In [2]:
import pandas as pd
import numpy as np

df_goodreads = pd.read_csv('data/books.csv', sep="\t", index_col=0)
df_goodreads.head(2)

,gr_book_id,gr_best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,title,...,ratings_count,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
book_id,,,,,,,,,,,,,,,,,,,,,
0,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,"The Hunger Games (The Hunger Games, #1)",...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
1,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,Harry Potter and the Sorcerer's Stone (Harry P...,...,4602479,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...


<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>

---

<a id="section11"></a> 
###  <font color="#00586D"> Lectura de las revisiones </font>
<br>

Los datos de la tabla apenas contienen información relativa al contenido de cada libro. Sin embargo, es posible acceder a los resúmenes almacenados en la propia web [GoodReads](www.goodreads.com) que, en cierto grado, aportan esta información. En el [anexo](#Anexo) se adjunta la función utilizada para obtenerlos. 

El resumen de cada libro ha sido almacenado en un archivo de texto denominado `./data/overviews/bookid.txt`, donde _bookid_ corresponde al identificador del libro (columna `book_id` de `df_goodreads`). 

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Implementar una función, denominada `get_overview(gr_book_id)`, que reciba el identificador de un libro, lea el archivo de texto que contiene el resumen correspondiente y lo devuelva en un _String_ (o devuelva `None` si este resumen no existe). 

In [3]:
import os

def get_overview(gr_book_id):
    
    if os.path.exists(f'./data/overviews/{gr_book_id}.txt'):
    
        with open(f'./data/overviews/{gr_book_id}.txt', encoding="utf8") as f:
            return f.read()
        
    return None
    
get_overview(320) # Cien años de soledad (Gabriel García Márquez)

'(Book Jacket Status: Jacketed)The brilliant, bestselling, landmark novel that tells the story of the Buendia family, and chronicles the irreconcilable conflict between the desire for solitude and the need for love—in rich, imaginative prose that has come to define an entire genre known as "magical realism."'

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Crear una columna en `df_goodreads`, denominada `overview`, que contenga la revisión del libro correspondiente. Rellenar los valores vacíos de esa columna con un _String_ de longitud cero ("").

In [4]:
df_goodreads['overview'] = df_goodreads.apply(lambda x: get_overview(x['gr_book_id']), axis=1).fillna("")
df_goodreads.head()

,gr_book_id,gr_best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,title,...,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url,overview
book_id,,,,,,,,,,,,,,,,,,,,,
0,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,"The Hunger Games (The Hunger Games, #1)",...,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...,Winning will make you famous. Losing means cer...
1,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,Harry Potter and the Sorcerer's Stone (Harry P...,...,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...,Harry Potter's life is miserable. His parents ...
2,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,"Twilight (Twilight, #1)",...,3916824,95009,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...,About three things I was absolutely positive.F...
3,2657,2657,3275794,487,61120081,9.780061e+12,Harper Lee,1960.0,To Kill a Mockingbird,To Kill a Mockingbird,...,3340896,72586,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...,The unforgettable novel of a childhood in a sl...
4,4671,4671,245494,1356,743273567,9.780743e+12,F. Scott Fitzgerald,1925.0,The Great Gatsby,The Great Gatsby,...,2773745,51992,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...,"On its first publication in 1925, The Great Ga..."


<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>

---

<a id="section12"></a> 
###  <font color="#00586D">Etiquetas </font>
<br>


En el conjunto de datos se proporcionan etiquetas relativas a cada libro que han sido aportadas por los usuarios. Esta información está incluída en dos archivos. El primero de ellos, `tags.csv`, contiene el identificador de cada etiqueta y el código correspondiente. El segundo, `book_tags.csv` contiene las etiquetas relativas a cada libro. Se almacenarán, respectivamente, en los _DataFrame_ `df_tags` y `df_book_tags`.

In [5]:
df_tags = pd.read_csv('./data/tags.csv')
df_book_tags = pd.read_csv('./data/book_tags.csv')

In [6]:
print("Etiquetas: ",df_tags.shape)
display(df_tags.iloc[2000:2002])
print("\n   -----------    \n")
print("Etiquetas/libro: ",df_book_tags.shape)
display(df_book_tags.head(3))

Etiquetas:  (34252, 2)


,tag_id,tag_name
2000,2000,alex-read
2001,2001,alex-rider



   -----------    

Etiquetas/libro:  (999912, 3)


,gr_book_id,tag_id,count
0,1,30574,167697
1,1,11305,37174
2,1,11557,34173


<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Incorporar el campo `tag_name` de `df_tags` a `df_book_tags`, sustituyendo el campo `tag_id` por la etiqueta (_String_) correspondiente. Para ello, fundir los dos _DataFrame_ y, posteriormente, eliminar las columnas `count` y `tag_id`.

In [7]:
df_book_tags = pd.merge(df_book_tags, df_tags, on='tag_id')
df_book_tags.drop(['count', 'tag_id'], axis=1, inplace=True)

df_book_tags.tail()

,gr_book_id,tag_name
999907,31538635,hogwarts
999908,32848471,jan-2017
999909,33288638,single-mom
999910,33288638,fave-author
999911,33288638,slowburn


<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Inspeccionar las etiquetas que aparecen en `df_book_tags` y el número de veces que aparece cada una. ¿Deberían eliminarse algunas?

In [8]:
df_book_tags.groupby('tag_name').size().sort_values().head(25)

tag_name
ｆａｖｏｕｒｉｔｅｓ                          1
queer-fantasy                       1
forensic-medicine-thriller          1
queen-elizabeth-ii                  1
queen-elizabeth                     1
forensic-sculptor                   1
quebec-city                         1
quarterbacks                        1
quanah-parker                       1
forest-fires                        1
quality-stories                     1
forest-of-hands-and-teeth-series    1
quality-childrens-books             1
forever-ago                         1
forever-amber                       1
quality                             1
quakers                             1
quaker                              1
format-graphic-novel                1
format-comic                        1
form-1                              1
pyramids                            1
pérez-reverte                       1
forgotten_realms                    1
forensic-archaeologist              1
dtype: int64

Deberían eliminarse aquellas etiquetas que sólo aparecen una vez.

Una de las cosas que se observan es que hay un alto número de etiquetas que aparecen una vez, y que son irrelevantes, por lo que es mejor ignorarlas. 


<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Eliminar todas las etiquetas que aparezcan menos de tres veces. 

<div class="alert alert-block alert-warning">
    
<i class="fa fa-exclamation-circle" aria-hidden="true"></i> Este ejercicio puede ser complejo, y hay varias formas de hacerlo. Una de ellas es agrupar por `tag_name`, mediante `groupby`, y filtrar con `filter` los grupos con tamaño >=3.
</div>

In [9]:
df_book_tags = df_book_tags.groupby('tag_name').filter(lambda x: len(x) >= 3)

df_book_tags.head()

,gr_book_id,tag_name
0,1,to-read
1,2,to-read
2,3,to-read
3,5,to-read
4,6,to-read


Otro de los problemas que se aprecian es que algunas etiquetas son genéricas, y no corresponden a libros concretos. Por ejemplo palabras como `read-readings` o `favourites`. 

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Eliminar todas las etiquetas que contengan estos términos (los términos de la lista `target_tags`).  

<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
En este caso, es recomendable no utilizar las etiquetas completas para eliminar así sus variantes también. 
</div>

In [10]:
target_tags = ['read', 'favo', 'own', 'top', 'book', 'librar']

df_book_tags = df_book_tags[~df_book_tags['tag_name'].str.contains('|'.join(target_tags))]
# df_book_tags = df_book_tags[df_book_tags['tag_name'].str.contains('|'.join(target_tags))==False]

df_book_tags.head()

,gr_book_id,tag_name
9983,1,fantasy
9984,2,fantasy
9985,3,fantasy
9986,5,fantasy
9987,6,fantasy


Por último,  se creará un `DataFrame` denominado `df_book_tag_text` en el que, para cada libro (indicado por su código `goodreads_book_id`), se añadirá una columna con _un solo campo de texto_, resultado de unir las etiquetas correspondientes.  

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Agrupar las entradas `DataFrame` en `df_book_tags` en función del campo `goodreads_book_id` y unir todas las etiquetas de cada grupo mediante `join`. Almacenar el resultado en un `DataFrame` denominado `df_book_tag_text`.

<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
Para pasos posteriores es necesario convertir el resultado resultante, una `Serie`, en un `DataFrame`, mediante `Series.to_frame()`. 
</div>

In [11]:
df_book_tag_text = df_book_tags.groupby('gr_book_id')['tag_name'].apply(', '.join).to_frame()
# df_book_tag_text = df_book_tags.groupby('gr_book_id').agg({'tag_name': lambda x: ', '.join(x)})

df_book_tag_text.head()

,tag_name
gr_book_id,
1,"fantasy, young-adult, fiction, harry-potter, y..."
2,"fantasy, young-adult, fiction, harry-potter, y..."
3,"fantasy, young-adult, fiction, harry-potter, y..."
5,"fantasy, young-adult, fiction, harry-potter, y..."
6,"fantasy, young-adult, fiction, harry-potter, y..."


<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Renombrar la columna `tag_name` de `df_book_tags` a `text_tags`. Rellenar los valores perdidos con un String vacío ("").

In [12]:
df_book_tag_text.rename(columns={'tag_name':'text_tags'}, inplace=True)
df_book_tag_text.fillna("", inplace=True)

df_book_tag_text.head()

,text_tags
gr_book_id,
1,"fantasy, young-adult, fiction, harry-potter, y..."
2,"fantasy, young-adult, fiction, harry-potter, y..."
3,"fantasy, young-adult, fiction, harry-potter, y..."
5,"fantasy, young-adult, fiction, harry-potter, y..."
6,"fantasy, young-adult, fiction, harry-potter, y..."


<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Incorporar la información del texto de las etiquetas, es decir, la columna `text_tags`, al `DataFrame` principal `df_goodreads` (utilizar _merge_).

In [13]:
df_goodreads = pd.merge(df_goodreads, df_book_tag_text, on='gr_book_id')

df_goodreads.head(2)

,gr_book_id,gr_best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,title,...,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url,overview,text_tags
0,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,"The Hunger Games (The Hunger Games, #1)",...,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...,Winning will make you famous. Losing means cer...,"fantasy, young-adult, fiction, ya, series, adv..."
1,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,Harry Potter and the Sorcerer's Stone (Harry P...,...,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...,Harry Potter's life is miserable. His parents ...,"fantasy, young-adult, fiction, harry-potter, y..."


<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="section13"></a> 
###  <font color="#00586D"> Preparación de datos </font>
<br>

Una vez incorporada la información relativa a resúmenes y etiquetas, esta se fundirá, constituyendo el contenido de cada libro. 


<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i></font> 
Fundir las columnas `overview` y `text_tags` de `df_goodreads` en otra columna llamada`text`. Eliminar las columnas en las que `text` tenga longitud 0.

In [14]:
df_goodreads['text'] = df_goodreads[['overview', 'text_tags']].apply(lambda x: ' '.join(x), axis=1)
df_goodreads = df_goodreads[df_goodreads['text'].apply(lambda x: len(x) > 0)]

df_goodreads.shape

(10000, 25)

####  <font color="#00586D"> Selección de cinco libros para pruebas </font>


<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Elegir cinco libros, y almacenarlos en el `DataFrame` `df_books_test` (copiadlos).

In [15]:
test_pos = np.array([2,4,7,9,12]) # Sugerencia. Podéis cambiarlos. 
# test_pos = np.array([13,115,129,292,3350]) # mi seleccion prueba

df_books_test = df_goodreads.iloc[test_pos]
df_books_test

,gr_book_id,gr_best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,title,...,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url,overview,text_tags,text
2,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,"Twilight (Twilight, #1)",...,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...,About three things I was absolutely positive.F...,"fantasy, young-adult, fiction, ya, sci-fi-fant...",About three things I was absolutely positive.F...
4,4671,4671,245494,1356,743273567,9.780743e+12,F. Scott Fitzgerald,1925.0,The Great Gatsby,The Great Gatsby,...,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...,"On its first publication in 1925, The Great Ga...","young-adult, fiction, default, classics, 5-sta...","On its first publication in 1925, The Great Ga..."
7,5107,5107,3036731,360,316769177,9.780317e+12,J.D. Salinger,1951.0,The Catcher in the Rye,The Catcher in the Rye,...,109383,185520,455042,661516,709176,https://images.gr-assets.com/books/1398034300m...,https://images.gr-assets.com/books/1398034300s...,The hero-narrator of The Catcher in the Rye is...,"young-adult, fiction, ya, default, classics, n...",The hero-narrator of The Catcher in the Rye is...
9,1885,1885,3060926,3455,679783261,9.780680e+12,Jane Austen,1813.0,Pride and Prejudice,Pride and Prejudice,...,54700,86485,284852,609755,1155673,https://images.gr-assets.com/books/1320399351m...,https://images.gr-assets.com/books/1320399351s...,"'It is a truth universally acknowledged, that ...","young-adult, fiction, default, classics, 5-sta...","'It is a truth universally acknowledged, that ..."
12,5470,5470,153313,995,451524934,9.780452e+12,"George Orwell, Erich Fromm, Celâl Üster",1949.0,Nineteen Eighty-Four,1984,...,41845,86425,324874,692021,908229,https://images.gr-assets.com/books/1348990566m...,https://images.gr-assets.com/books/1348990566s...,"Among the seminal texts of the 20th century, N...","fantasy, young-adult, fiction, sci-fi-fantasy,...","Among the seminal texts of the 20th century, N..."


<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>

---

<a id="section2"></a> 
## <font color="#00586D"> 2. Limpieza del texto </font>
<br>

En este proyecto, el contenido (texto) asociado a cada libro se representa mediante un modelo de bolsa de palabras. Como se ha explicado a lo largo del módulo, cuando se trabaja con este tipo de representación es recomendable limpiar el texto y eliminar información irrelevante. 

En este caso se procederá de manera similar a como se ha hecho en las prácticas anteriores, es decir, se eliminarán palabras de menos de tres letras, se eliminarán _stop words_ y se utilizará lematización. Además se eliminarán nombres de entidades, es decir, nombres propios. Estos pueden hacer que el sistema discrimine en base en términos muy específicos, pero irrelevantes, como por ejemplo nombres de ciudades. 

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> Hemos optado por eliminar entidades, en parte, para que veáis como se hace. Se puede trabajar de otro modo, no eliminando la entidades, y luego filtrando por _IDF_ para que solamente permanezcan las importantes. Por ejemplo el transformador eliminaría palabras como "Albacete" si aparece en uno o dos libros, y dejaría "Nueva York" si es suficientemente importante porque aparece en varios.
</div>

A continuación se describe cada uno de los pasos.  En primer lugar, se utilizará un objeto `nltk.tokenize.RegexpTokenizer` para separar las frases en _tokens_ compuestos de al menos tres caracteres alfabéticos. 

In [16]:
import nltk
#nltk.download('wordnet')
#nltk.download('averaged_perceptron_tagger')
#nltk.download('maxent_ne_chunker')
#nltk.download('words')

from nltk.tokenize import RegexpTokenizer

tokenizer = RegexpTokenizer(r'[a-zA-Z][a-zA-Z][a-zA-Z]+') 

Posteriormente, es necesario almacenar las _stop words_ en una lista, denominada `stopwords_en`.

In [17]:
from nltk.corpus import stopwords

stopwords_en = stopwords.words('english')
print(stopwords_en[:10])

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're"]


Por último, se sustituirá cada palabra por su lema (también se ha visto en las prácticas anteriores) mediante el objeto `nltk.stem.wordnet.WordNetLemmatizer`.

In [18]:
from nltk.stem.wordnet import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

El _reconocimiento de entidades nombradas_ consiste en categorizar, dentro de un texto, las palabras o n-gramas que representan entidades que pueden ser reconocidas, tales como personas, localizaciones, etc. Aunque esta tarea implica en otros contextos devolver información con respecto a estas entidades (a qué corresponden), en el ámbito de este proyecto simplemente se eliminarán las entidades.

El reconocimiento de entidades requiere que se lleve a cabo un etiquetado POS (_Part Of Speech tagging_) que básicamente asigna a cada token la función sintáctica dentro del documento. La siguiente función elimina las entidades de un texto pasado como argumento.

In [19]:
from nltk import pos_tag
from nltk.tree import Tree

def remove_entity(text):
    tokenized_text = nltk.word_tokenize(text)
    tagged =  nltk.pos_tag(tokenized_text)
    chunked = nltk.ne_chunk(tagged, binary=True)
    new_text = []
    entities = []
    for i in chunked:
        if not type(i) == Tree:
            new_text.append(i[0])
        else:
            entities.append(i)
    return " ".join(new_text)

text = df_goodreads.iloc[0]['text']
print(text[:200])
t = remove_entity(text)
print("\n",t[:200])

Winning will make you famous. Losing means certain death.The nation of Panem, formed from a post-apocalyptic North America, is a country that consists of a wealthy Capitol region surrounded by 12 poor

 Winning will make you famous . Losing means certain death.The nation of , formed from a post-apocalyptic , is a country that consists of a wealthy Capitol region surrounded by 12 poorer districts . Ea


<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Implementar una función, denominada `clean` que tome como entrada un texto, y devuelva una versión "limpia" del mismo, es decir, de la que se hayan eliminado _stop words_ y entidades, y sustituido cada palabra por un lema. 

In [20]:
def clean(text):
    
    no_entity_text = remove_entity(text)
    tokens = [lemmatizer.lemmatize(token) for token in tokenizer.tokenize(no_entity_text) if token not in stopwords_en]
    
    return " ".join(tokens)   

clean(text)[:200]

'Winning make famous Losing mean certain death The nation formed post apocalyptic country consists wealthy Capitol region surrounded poorer district Early history rebellion led district resulted destru'

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Limpiar los datos almacenados en la columna `text` del `DataFrame` `goodreads` y almacenar el resultado en una serie denominada `data_books` (requiere varios minutos).

In [21]:
data_books = df_goodreads['text'].apply(clean)

In [22]:
data_books

0       Winning make famous Losing mean certain death ...
1       life miserable His parent dead stuck heartless...
2       About three thing absolutely positive First va...
3       The unforgettable novel childhood sleepy town ...
4       first publication The largely dismissed light ...
                              ...                        
9995    The Edge lie world border people shop magic fa...
9996    Caro life began greatly acclaimed The also win...
9997    Brian Aubrey Maturin volume actually constitut...
9998    The acclaimed author groundbreaking bestseller...
9999    The First World War created modern world confl...
Name: text, Length: 10000, dtype: object

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>


---

<a id="section3"></a> 
## <font color="#00586D"> 3. Búsqueda por similaridad de la representación tf-idf </font>
<br>

La información _tf-idf_ es muy útil de cara a clasificar y comparar documentos. Cada texto se representa  mediante un vector de valores _tf-idf_, y la búsqueda de documentos se apoya en una medida de similaridad, la _similaridad coseno_, para esta representación.

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i></font> Crear un objeto `TfidfVectorizer` de `sklearn`, denominado `tfidf_vect`, que represente un máximo de 20000 términos. Obtener la matriz _tf-idf_ de los datos almacenados en `data_books` y almacenarla en una variable denominada `data_books_tfidf`. Extraer los términos considerados en `tfidf_vect`, y almacenarlos en la variable `terms`.

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vect = TfidfVectorizer(max_features=20000)
data_books_tfidf = tfidf_vect.fit_transform(data_books)
terms = tfidf_vect.get_feature_names_out()

In [24]:
print(terms[0:50])

['aaa' 'aaronovitch' 'ab' 'abandon' 'abandonados' 'abandoned' 'abandoning'
 'abandonment' 'abarat' 'abbey' 'abbi' 'abbott' 'abby' 'abc' 'abduct'
 'abducted' 'abduction' 'abductor' 'abducts' 'aber' 'abercrombie' 'abhors'
 'abhorsen' 'abide' 'abiding' 'ability' 'abject' 'able' 'ablo' 'ably'
 'aboard' 'abolition' 'abolitionist' 'abomination' 'aboriginal' 'abortion'
 'abound' 'abounds' 'about' 'above' 'abraham' 'abrasive' 'abridged'
 'abridgment' 'abroad' 'abrupt' 'abruptly' 'absence' 'absent' 'absolute']


In [25]:
print(terms[19950:])

['youtubers' 'yuki' 'yum' 'yummy' 'zac' 'zach' 'zachary' 'zadie' 'zaf'
 'zafon' 'zahn' 'zakaria' 'zan' 'zany' 'zapata' 'zara' 'zarathustra'
 'zeal' 'zealand' 'zealot' 'zealous' 'zebra' 'zedan' 'zeit' 'zeitgeist'
 'zelazny' 'zelinsky' 'zen' 'zeno' 'zero' 'zidan' 'ziegesar' 'zijn'
 'zimbabwe' 'zimmer' 'zip' 'zodiac' 'zoe' 'zoey' 'zola' 'zombie' 'zone'
 'zoo' 'zoology' 'zoom' 'zorba' 'zuckerberg' 'zum' 'zur' 'zwei']


#### <font color="#00586D"> Búsqueda del libro más similar </font>

Se tomará un libro incluído en el conjunto de datos `df_books_test`. Por ejemplo, el primero.

In [26]:
print(df_books_test.iloc[0])
text_query = df_books_test.iloc[0].text
text_query

gr_book_id                                                               41865
gr_best_book_id                                                          41865
work_id                                                                3212258
books_count                                                                226
isbn                                                                 316015849
isbn13                                                         9780316015840.0
authors                                                        Stephenie Meyer
original_publication_year                                               2005.0
original_title                                                        Twilight
title                                                  Twilight (Twilight, #1)
language_code                                                            en-US
average_rating                                                            3.57
ratings_count                                       

"About three things I was absolutely positive.First, Edward was a vampire.Second, there was a part of him—and I didn't know how dominant that part might be—that thirsted for my blood.And third, I was unconditionally and irrevocably in love with him.In the first book of the Twilight Saga, internationally bestselling author Stephenie Meyer introduces Bella Swan and Edward Cullen, a pair of star-crossed lovers whose forbidden relationship ripens against the backdrop of small-town suspicion and a mysterious coven of vampires. This is a love story with bite. fantasy, young-adult, fiction, ya, sci-fi-fantasy, default, 5-stars, novels, fantasy-sci-fi, paranormal, ya-fantasy, teen, english, urban-fantasy, supernatural, ya-fiction, young-adult-fiction, scifi-fantasy, faves, kindle, contemporary, on-my-shelf, have, movie, finished, sci-fi, ya-lit, werewolves, séries, love, science-fiction, finished-series, youngadult, abandoned, did-not-finish, dnf, american, movies, drama, high-school, romantic

In [27]:
clean(text_query)

'About three thing absolutely positive First vampire Second part know dominant part might thirsted blood And third unconditionally irrevocably love first book internationally bestselling author introduces pair star crossed lover whose forbidden relationship ripens backdrop small town suspicion mysterious coven vampire This love story bite fantasy young adult fiction sci fantasy default star novel fantasy sci paranormal fantasy teen english urban fantasy supernatural fiction young adult fiction scifi fantasy faves kindle contemporary shelf movie finished sci lit werewolf ries love science fiction finished series youngadult abandoned finish dnf american movie drama high school romantic horror chick lit first series teen fiction meh young adult guilty pleasure guilty pleasure paranormal romance paranormal romance vampire vampire completed series never vamp paranormal fantasy fantasy paranormal pnr love triangle twilight stephenie meyer twilight saga twilight series stephanie meyer'

Para poder buscar otro libro por similaridad, es necesario transformar el texto de consulta a la representación _tf-idf_. Previamente, ha de ser preprocesado (limpiado con la función `clean`).

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Transformar el documento `text_query` a formato _tf-idf_ y almacenar el resultado en `text_query_tfidf`.

<div class="alert alert-block alert-warning">
    
<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
Esta sería la manera de proceder con libros nuevos. No obstante, en este caso concreto, ___como el libro ya estaba en el corpus y se ha obtenido su representación___, se podría obtener el vector directamente desde `data_books_tfidf`.

</div>

In [28]:
# Se obtiene el índice del libro en df_goodreads
book_index = df_goodreads[df_goodreads['gr_book_id']==df_books_test.iloc[0]['gr_book_id']].index[0]
book_index

2

In [29]:
text_query_tfidf = data_books_tfidf[book_index]

In [30]:
print(text_query_tfidf)

  (0, 16926)	0.11648725927542602
  (0, 15334)	0.07946222876540832
  (0, 11182)	0.25550683061055446
  (0, 16928)	0.1345105748631428
  (0, 18401)	0.36234020954890356
  (0, 13304)	0.07252922490524884
  (0, 19029)	0.07757216368770502
  (0, 11954)	0.047905915366734345
  (0, 13267)	0.12871572171716614
  (0, 7777)	0.13244353701857978
  (0, 11002)	0.07491805555751396
  (0, 2868)	0.04815504843858306
  (0, 8387)	0.0527784085029798
  (0, 15156)	0.06333248781765508
  (0, 15562)	0.039430411156412826
  (0, 556)	0.041953602204633754
  (0, 5209)	0.04260126710751406
  (0, 6753)	0.034685033799773585
  (0, 5)	0.03416142717010328
  (0, 19938)	0.07693845987755078
  (0, 15021)	0.09271687319001347
  (0, 19495)	0.08100551469371729
  (0, 11665)	0.13249323436046564
  (0, 1816)	0.1233724722390447
  (0, 17008)	0.037684284351522775
  :	:
  (0, 11761)	0.06854822147184134
  (0, 18259)	0.07544368581054438
  (0, 3501)	0.07167439220082177
  (0, 5349)	0.0393209465913237
  (0, 15581)	0.039815804659290306
  (0, 10348)	0.1

In [31]:
print(f'Dimensión:  {text_query_tfidf.toarray()[0].shape}')

Dimensión:  (20000,)


In [32]:
book_tfidf = dict(zip(terms, text_query_tfidf.toarray()[0]))
{k:v for k,v in book_tfidf.items() if v>0}

{'abandoned': 0.03416142717010328,
 'about': 0.11983752533493759,
 'absolutely': 0.11938378402569855,
 'adult': 0.06109380818620229,
 'american': 0.041953602204633754,
 'and': 0.04685508667233666,
 'author': 0.038861462012976446,
 'backdrop': 0.1082864707399075,
 'bestselling': 0.06333248781765508,
 'bite': 0.1233724722390447,
 'blood': 0.07822850450640377,
 'book': 0.05019669928720415,
 'chick': 0.04815504843858306,
 'completed': 0.07167439220082177,
 'contemporary': 0.02913723792776473,
 'coven': 0.14196123519041365,
 'crossed': 0.11466640553891197,
 'default': 0.02193466068493513,
 'dnf': 0.04260126710751406,
 'dominant': 0.1316779652707426,
 'drama': 0.0393209465913237,
 'english': 0.03630299968985666,
 'fantasy': 0.269455102714907,
 'faves': 0.05903790908473825,
 'fiction': 0.0919422774587508,
 'finish': 0.034685033799773585,
 'finished': 0.07555822543617036,
 'first': 0.12239257075106352,
 'forbidden': 0.08809882245514286,
 'guilty': 0.13244353701857978,
 'high': 0.04472444505194

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Mostrar los 10 términos más relevantes (con mayor valor _tf-idf_) para el documento `text_query`.

In [33]:
relevant_terms = sorted(book_tfidf, key=book_tfidf.get, reverse=True)[:10]
# relevant_terms = sorted(book_tfidf.items(), key=lambda item: item[1], reverse=True)[:10]
relevant_terms

['twilight',
 'vampire',
 'fantasy',
 'meyer',
 'paranormal',
 'unconditionally',
 'love',
 'coven',
 'stephenie',
 'movie']

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Calcular las similaridades (similaridad coseno) del documento `text_query_tfidf` con el resto de documentos en la matriz `data_books_tfidf`, y almacenarlas en un vector denominado `similarities_tfidf`.

<div class="alert alert-block alert-warning">
    
<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
La función `cosine_similarity` toma como argumentos dos matrices y devuelve una matriz (aunque sea unidimensional). Para obtener un vector unidemensional, y operar posteriormente con más agilidad, `similarities_tfidf` ha de almacenar el elemento cero de la matriz devuelta por la función, o transformar esta matriz en un vector. 
</div>

In [34]:
from sklearn.metrics.pairwise import  cosine_similarity

similarities_tfidf = cosine_similarity(text_query_tfidf, data_books_tfidf)
similarities_tfidf

array([[0.12929952, 0.12712662, 1.        , ..., 0.03072685, 0.01977996,
        0.0199855 ]])

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Obtener los índices de los 6 libros más similares a `text_query` (obviamente, el más similar será el propio libro).



In [35]:
top_similarities = similarities_tfidf[0].argsort()[::-1][0:6] 
df_goodreads.iloc[top_similarities]['title']

2                                 Twilight (Twilight, #1)
51                                 Eclipse (Twilight, #3)
2020             The Twilight Collection (Twilight, #1-3)
48                                New Moon (Twilight, #2)
1618    The Twilight Saga Complete Collection  (Twilig...
833                         Midnight Sun (Twilight, #1.5)
Name: title, dtype: object

Se aprecia que, efectivamente, los libros son muy similares. De hecho, todos corresponden a la saga. Esto puede deverse al uso de palabras clave muy concretas como _"Twilight"_.

<font color="#004D7F"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i></font> Guardar la matriz `data_books_tfidf` en el archivo `./data/data_books_tfidf.p` (para uso posterior).

In [36]:
import pickle

with open("./data/data_books_tfidf.pickle", "bw") as f:
    f.write(pickle.dumps(data_books_tfidf))

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>


---

<a id="section4"></a> 
## <font color="#00586D"> 4. Recomendación basada en contenido </font>
<br>

Existen muchas posibilidades en la implementación de sistemas de recomendación basados en contenido. En general, todas requieren:

* Determinar qué ítems prefiere el usuario
* Encontrar ítems similares
* Priorizar esos ítems


Para ilustrar el funcionamiento de este tipo de sistemas, se tomarán los datos relativos a la actividad de un usuario escogido al azar, y se devolverá un conjunto de libros recomendados. 


El archivo `'./data/ratings.csv` contiene las valoraciones hechas por más de 53000 usuarios a los 10000 libros. En total, contiene cerca de un millón de entradas en formato (`user_id`, `book_id`, `rating`). El tamaño de la base de datos dificulta el trabajo con una matriz, aunque sea dispersa, ya que el proceso de elaboración (mediante `pivot` es muy lento). 

In [37]:
df_ratings = pd.read_csv('./data/ratings.csv', sep=',')

display(df_ratings.head())
df_ratings.shape

,user_id,book_id,rating
0,313,0,5
1,438,0,3
2,587,0,5
3,1168,0,4
4,1184,0,4


(981756, 3)

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>


---

<a id="section41"></a> 
### <font color="#00586D"> Selección de libros preferidos por el usuario </font>
<br>

El primer paso de la recomendación consiste en determinar qué libros prefiere el usuario.  Una posibilidad consiste en seleccionar aquellos para los que éste ha otorgado una puntuación mayor que tres. 

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Tomar un usuario al azar y devolver las entradas correspondientes a los libros que ha votado. Almacenarlas en el `DataFrame` `ratings_user`. Descartar de `ratings_user` todos los libros con puntuación menor que cuatro. 

In [38]:
np.random.seed(0)
test_user = np.random.randint(max(df_ratings['user_id']))

ratings_user = df_ratings[(df_ratings['user_id']==test_user) & (df_ratings['rating']>=4)]
ratings_user

,user_id,book_id,rating
53101,2732,531,5
209776,2732,2099,5
214080,2732,2142,5
242344,2732,2425,4
369792,2732,3702,4
401262,2732,4018,4
425738,2732,4264,5
476351,2732,4773,4
524632,2732,5262,5
543609,2732,5454,4


<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>


---

<a id="section42"></a> 
### <font color="#00586D"> Búsqueda de libros similares </font>
<br>

Una vez se dispone de la lista de libros, es necesario encontrar libros similares. 

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i> </font>Extraer la representación de los libros relativos al usuario y copiarla en una matriz denominada `user_books_tfidf`. 

In [39]:
books_user_id = ratings_user['book_id']
user_books_tfidf = data_books_tfidf[books_user_id]
user_books_tfidf.shape

(14, 20000)

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i> </font> Calcular la matriz de similaridades coseno entre `user_books_tfidf`, `data_books_tfidf`. Fijar las similaridades de los libros leídos con el usuario  (`books_user_id `) consigo mismos a -1 con el fin de descartarlos. 

In [40]:
similarities_user = cosine_similarity(user_books_tfidf, data_books_tfidf)

# Se descartan los libros que ya se han leído
for i in range(len(books_user_id)):
    similarities_user[i][books_user_id.tolist()]=-1

similarities_user.shape

(14, 10000)

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i> </font> Obtener los 3 libros más similares _a cada uno_ de los leídos por el usuario y almacenarlos en la matriz `similar_books`.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> La función `similarities_user.argsort(axis=1)` devuelve los índices ordenados (ascendentemente) para cada fila del array. Se trata de coger las 3 últimas columnas de `similarities_user`. 
</div>

In [41]:
similar_books = similarities_user.argsort(axis=1)[:,-3:]

print(similar_books, "("+str(len(similar_books))+" libros distintos.)")

[[ 734 2842 6774]
 [8419 6889 4950]
 [7071 1622 9600]
 [8700 3678 6444]
 [1464 7578 7793]
 [4172 9955 9778]
 [ 846 9976 1674]
 [9211 8379 2022]
 [4252 6335 7087]
 [2474  387 6754]
 [1687 8349 5186]
 [3027 8626 1956]
 [ 175 1074  304]
 [1347  357 3273]] (14 libros distintos.)


<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>


---

<a id="section43"></a> 
### <font color="#00586D"> Priorización de resultados y recomendación</font>

Llegados a este punto, se han obtenido los libros más similares a los leídos y valorados positivamente por el usuario. Una posibilidad a la hora de priorizar los resultados consistiría en dar más peso a aquellos libros similares que han aparecido más veces en la matriz `similar_books`. 

También es posible utilizar la información del `DataFrame` `df_goodreads` para priorizar los libros. En concreto, la columna `average_rating` contiene la valoración media de cada libro en la plataforma, y puede ser utilizada para priorizar. 

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i> </font> Obtener los 10 libros con mayor `average_rating` de entre los más similares a los valorados positivamente por el usuario (`similar_books`). Almacenar sus índices en un array denominado `rec_books_user_id`.

In [42]:
similar_books_flatten = list(set(similar_books.flatten()))

In [43]:
rec_books_user_id = df_goodreads.iloc[similar_books_flatten].nlargest(10, 'average_rating').index.values.tolist()
rec_books_user_id

[3678, 6444, 734, 357, 1622, 6889, 8700, 9955, 9211, 175]

Por último, se imprimen los nombres de los libros preferidos por el usuario, cuyos índices están almacenados en `books_user_id`, y la recomendación, cuyos índices están almacenados en `rec_books_user_id`.

In [44]:
print("Libros preferidos por el usuario: ")
print("----------------------------------")
for book in df_goodreads.iloc[books_user_id][['title', 'authors']].itertuples():
    print(book.authors, "\n\t",book.title)
    
print("\n\nRecomendación: ")
print("----------------------------------")
for book in df_goodreads.iloc[rec_books_user_id][['title', 'authors']].itertuples():
    print(book.authors, "\n\t",book.title)   

Libros preferidos por el usuario: 
----------------------------------
Frank Miller, David Mazzucchelli, Richmond Lewis, Dennis O'Neil 
	 Batman: Year One
Frank Miller 
	 Batman: The Dark Knight Returns #1
Mark Waid, Alex Ross, Elliot S. Maggin 
	 Kingdom Come
Frank Miller, Lynn Varley 
	 300
Philip Roth 
	 The Human Stain (The American Trilogy, #3)
Kurt Busiek, Alex Ross 
	 Marvels
A. Manette Ansay 
	 Vinegar Hill
Michael Blake 
	 Dances with Wolves (Dances with Wolves, #1)
Alexander McCall Smith 
	 The Full Cupboard of Life (No. 1 Ladies' Detective Agency, #5)
Gary Paulsen 
	 Brian's Winter (Brian's Saga, #3)
Richard Russo 
	 Bridge of Sighs
Tracy Chevalier 
	 Falling Angels
Stephen King 
	 Apt Pupil
Dr. Seuss 
	 And to Think That I Saw it on Mulberry Street


Recomendación: 
----------------------------------
Bryan Lee O'Malley 
	 Scott Pilgrim, Volume 6: Scott Pilgrim's Finest Hour
Katsuhiro Otomo, Yoko Umezawa, Linda M. York, Jo Duffy 
	 Akira, Vol. 1
Alan Moore, Brian Bolland, Tim

Puede observarse que, efectivamente, los títulos recomendados guardan mucha relación con los títulos valorados positivamente por el usuario. 

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>


---

<a id="section5"></a> 
##  <font color="#00586D"> 5. Sistema híbrido de recomendación  </font>
<br>

<br> En el apartado anterior se ha utilizado la valoración media de cada libro para priorizar los vecinos más cercanos. Existe otra posibilidad, que consite en utilizar las valoraciones de los libros obtenidas mediante filtrado colaborativo. Esto constituye un sistema híbrido, ya que se mezclan las dos aproximaciones. Por una parte, se seleccionan libros basados en contenido, y de ellos, se muestra el conjunto para el que se predice una mayor puntuación por parte del usuario.
<br>

No existe mucho material disponible en relación a este tipo de modelos. Tampoco una librería de referencia, aunque va cobrando popularidad [surprise](http://surpriselib.com/). En primer lugar, se utilizará esta librería para obtener los scores mediante filtrado colaborativo. En concreto, se utilizará el algoritmo SVD (visto en clase). 

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> El autor principal de la librería describe en en una serie de artículos [enlace](http://nicolas-hug.com/blog/) como funciona la versión básica de este algoritmo. La lectura es muy recomendable. 
</div>

In [45]:
#!pip install surprise

En la siguiente celda, se cargan los votos  en la estructura `ratings_data` (usada por `surprise`) y se aprende un modelo SVD.

In [46]:
from surprise import SVD, Dataset, Reader
from surprise.model_selection.split import train_test_split

reader = Reader()
ratings_data = Dataset.load_from_df(df_ratings, reader).build_full_trainset()
svd = SVD(n_factors=10)
svd.fit(ratings_data);

In [47]:
similar_books

array([[ 734, 2842, 6774],
       [8419, 6889, 4950],
       [7071, 1622, 9600],
       [8700, 3678, 6444],
       [1464, 7578, 7793],
       [4172, 9955, 9778],
       [ 846, 9976, 1674],
       [9211, 8379, 2022],
       [4252, 6335, 7087],
       [2474,  387, 6754],
       [1687, 8349, 5186],
       [3027, 8626, 1956],
       [ 175, 1074,  304],
       [1347,  357, 3273]], dtype=int64)

In [48]:
scores = np.zeros(similar_books.shape)
scores

array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]])

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i> </font> Obtener uno a uno, las valoraciones de los libros similares a los preferidos por el usuario (`similar_books`), y tomar los 10 con más valoración. 

In [49]:
for i in range (similar_books.shape[0]):
    for j in range (similar_books.shape[1]):
        scores[i,j]=svd.predict(uid=test_user, iid=similar_books[i,j]).est
scores

array([[4.36772028, 3.80855372, 4.1270203 ],
       [3.93040973, 4.07753733, 4.03245859],
       [3.88855458, 4.32173147, 4.21417654],
       [3.96340688, 4.36664035, 4.3470056 ],
       [4.00318237, 3.83992417, 3.5567432 ],
       [3.90516236, 4.28250197, 4.06667444],
       [3.74110484, 3.75337639, 3.64616196],
       [4.05815822, 3.40681339, 3.89526475],
       [3.97435762, 3.97192137, 3.91614659],
       [4.14127198, 3.74793836, 3.49117205],
       [3.72515475, 3.59605579, 4.2977505 ],
       [3.89976801, 4.11563368, 4.02812685],
       [3.9752987 , 3.46497766, 3.55244877],
       [4.02630686, 4.20169655, 3.93435514]])

In [50]:
rec_books_user_id =similar_books.flatten()[scores.flatten().argsort()[:-11:-1]]
rec_books_user_id

array([ 734, 3678, 6444, 1622, 5186, 9955, 9600,  357, 2474, 6774],
      dtype=int64)

In [51]:
print("\n\nRecomendación: ")
print("----------------------------------")
for book in df_goodreads.iloc[rec_books_user_id][['title', 'authors']].itertuples():
    print(book.authors, "\n\t",book.title)    



Recomendación: 
----------------------------------
Alan Moore, Brian Bolland, Tim Sale 
	 Batman: The Killing Joke
Bryan Lee O'Malley 
	 Scott Pilgrim, Volume 6: Scott Pilgrim's Finest Hour
Katsuhiro Otomo, Yoko Umezawa, Linda M. York, Jo Duffy 
	 Akira, Vol. 1
Jeph Loeb, Tim Sale, Gregory Wright, Richard Starkings 
	 Batman: The Long Halloween
Richard Russo 
	 Nobody's Fool
Joss Whedon, John Cassaday 
	 Astonishing X-Men, Volume 2: Dangerous
Alan Moore, Curt Swan, George Pérez, Kurt Schaffenberger 
	 Superman: Whatever Happened to the Man of Tomorrow?
Dr. Seuss 
	 Oh, The Places You'll Go!
Jean Craighead George 
	 My Side of the Mountain (Mountain, #1)
Brian K. Vaughan, Adrian Alphona 
	 Runaways, Vol. 1: Pride and Joy (Runaways, #1)


<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>


---

<a id="section6"></a> 
## <font color="#00586D"> 6. LDA y búsqueda por similaridad </font>
<br>

LDA permite modelar los grupos que dan lugar al corpus de documentos, de modo que cada documento se representa como una mixtura de las distribuciones de probabilidad de cada grupo, dada por un vector de pesos. Esta representación puede ser utilizada también para caracterizar el documento (en lugar de _tf-idf_) y permite cuantificar la similaridad con otros. Por tanto, también serviría para implementar un sistema de recomendación (basado en contenido o híbrido) similar al anterior.

LDA trabaja con la matriz de frecuencias de  términos por documento (_term frecuency_). Por tanto, es necesario primero obtenerla. 

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Construir un objeto `CountVectorizer` de `sklearn` y almacenarlo en la variable `cv_vect`.  Obtener la matriz `data_books_cv` a partir de los datos almacenados en `data_books`. Utilizar un tamaño máximo de vocabulario de 5000 (para que LDA no requiera demasiado tiempo y memoria). Almacenar el vocabulario generado en `cv_vect` en la variable `terms`. 

In [52]:
from sklearn.feature_extraction.text import CountVectorizer

cv_vect = CountVectorizer(max_features=5000)
data_books_cv = cv_vect.fit_transform(data_books)
terms = cv_vect.get_feature_names()

Una vez construida la matriz de frecuencias, se ejecutará el algoritmo LDA, para representar los documentos mediante 20 grupos.

In [53]:
from sklearn.decomposition import LatentDirichletAllocation

# Número de temas
n_topics = 20

lda = LatentDirichletAllocation(n_components=n_topics, max_iter=20, learning_method='online',
                                learning_offset=50., random_state=0, n_jobs=-1)

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Construir el modelo LDA y almacenarlo en la variable `topic_term`. Guardar la representación de los documentos en la variable `data_books_lda`.

In [54]:
from time import time
start = time()
data_books_lda = lda.fit_transform(data_books_cv)
topic_term = lda.components_

print("Tiempo: {:0.3f}s.".format(time() - start))
print("Tamaño del modelo:", topic_term.shape)
print("Tamaño del resultado de la transformación: ", data_books_lda.shape)

Tiempo: 71.021s.
Tamaño del modelo: (20, 5000)
Tamaño del resultado de la transformación:  (10000, 20)


La siguiente función (de la práctica de LDA) imprime los términos y scores más importantes para cada grupo.

In [55]:
def topic_relevant_words(topic_id, model, terms, num_words):
    print("Grupo: ", topic_id)
    print("----------------------")
    # Extrae las posiciones de los mayores scores. 
    top_term_ids = model[topic_id,:].argsort()[:-num_words-1:-1]
    # Extrae los términos de las posiciones correspondientes
    top_terms = [terms[id_term] for id_term in top_term_ids]
    # Extrae y calcula los scores
    top_scores =  model[topic_id,top_term_ids] / np.sum(model[topic_id,:])
    # Imprime los resultados
    for term, score in zip(top_terms,top_scores):
        print("{:s}:     \t{:.4f}".format(term,score))
    print("----------------------\n")

topic_relevant_words(0,topic_term, terms, 10)

Grupo:  0
----------------------
nobel:     	0.0476
prize:     	0.0460
literature:     	0.0405
latin:     	0.0390
fiction:     	0.0342
american:     	0.0320
realism:     	0.0271
novel:     	0.0260
spanish:     	0.0243
literatura:     	0.0226
----------------------



#### <font color="#00586D"> Búsqueda del libro más similar </font>

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font>  Obtener la representación a partir del modelo `lda` del documento `text_query`. Almacenarla en la variable `text_query_lda`. 

In [56]:
# Se vectoriza
vect_text = cv_vect.transform([text_query])
vect_text

<1x5000 sparse matrix of type '<class 'numpy.int64'>'
	with 90 stored elements in Compressed Sparse Row format>

In [57]:
# Se extraen las probabilidades
text_query_lda = lda.transform(vect_text)
text_query_lda

array([[3.49650355e-04, 3.56554057e-01, 3.49650351e-04, 3.49650358e-04,
        5.99965940e-02, 3.49650351e-04, 3.49650356e-04, 9.09863086e-02,
        3.49650352e-04, 3.73926423e-01, 3.49650356e-04, 3.49650359e-04,
        3.49650350e-04, 3.49650357e-04, 3.49650355e-04, 1.21776506e-02,
        1.01463861e-01, 3.49650353e-04, 3.49650350e-04, 3.49650354e-04]])

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Mostrar las características de los tres grupos más relevantes en el texto `text_query`. 

In [58]:
topics = text_query_lda.argsort()[0][:-4:-1]
topics

array([ 9,  1, 16], dtype=int64)

In [59]:
for id_topic in topics:
    topic_relevant_words(id_topic,topic_term, terms, 10)

Grupo:  9
----------------------
fiction:     	0.0297
adult:     	0.0193
contemporary:     	0.0190
romance:     	0.0156
love:     	0.0138
lit:     	0.0115
young:     	0.0113
star:     	0.0104
life:     	0.0100
buy:     	0.0099
----------------------

Grupo:  1
----------------------
paranormal:     	0.0692
fantasy:     	0.0623
series:     	0.0383
vampire:     	0.0302
horror:     	0.0274
romance:     	0.0268
supernatural:     	0.0245
adult:     	0.0194
urban:     	0.0187
fiction:     	0.0127
----------------------

Grupo:  16
----------------------
art:     	0.0712
the:     	0.0320
movie:     	0.0243
book:     	0.0208
history:     	0.0165
artist:     	0.0152
buy:     	0.0116
design:     	0.0110
film:     	0.0108
non:     	0.0102
----------------------



<font color="#004D7F"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i></font> Calcular las similaridades (distancia_euclídea) del documento `text_query_lda` con el resto de documentos en la matriz `data_books_lda`, y almacenarlas en un vector denominado `similarities_lda`.

In [60]:
text_query_lda

array([[3.49650355e-04, 3.56554057e-01, 3.49650351e-04, 3.49650358e-04,
        5.99965940e-02, 3.49650351e-04, 3.49650356e-04, 9.09863086e-02,
        3.49650352e-04, 3.73926423e-01, 3.49650356e-04, 3.49650359e-04,
        3.49650350e-04, 3.49650357e-04, 3.49650355e-04, 1.21776506e-02,
        1.01463861e-01, 3.49650353e-04, 3.49650350e-04, 3.49650354e-04]])

In [61]:
text_query_lda.shape

(1, 20)

In [62]:
data_books_lda

array([[3.35570472e-04, 3.35570476e-04, 3.35570472e-04, ...,
        3.35570476e-04, 3.35570470e-04, 1.79612165e-02],
       [2.87356325e-04, 2.87356327e-04, 2.87356324e-04, ...,
        2.87356326e-04, 2.87356324e-04, 2.87356328e-04],
       [3.87596906e-04, 3.90348251e-01, 3.87596902e-04, ...,
        3.87596903e-04, 3.87596899e-04, 3.87596903e-04],
       ...,
       [4.09836069e-04, 4.09836074e-04, 4.09836068e-04, ...,
        4.09836074e-04, 4.09836066e-04, 4.09836073e-04],
       [3.06748468e-04, 3.06748471e-04, 3.06748467e-04, ...,
        1.00855967e-01, 3.06748466e-04, 3.06748470e-04],
       [1.92307694e-04, 1.92307695e-04, 1.92307692e-04, ...,
        6.47341957e-01, 1.92307693e-04, 1.92307709e-04]])

In [63]:
data_books_lda.shape

(10000, 20)

In [64]:
from sklearn.metrics.pairwise import euclidean_distances

similarities_lda = euclidean_distances(text_query_lda, data_books_lda)

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Obtener los índices de los 10 libros más similares a `text_query`. Hay que tener en cuenta que `euclidean_distances` devuelve distancias, no similaridades, por lo que hay que tomar los primeros índices. 

In [65]:
top_similarities = similarities_lda.argsort()[0,0:10]

df_goodreads.iloc[top_similarities]['title']

5728                            Nevermore (Nevermore, #1)
181           Beautiful Creatures (Caster Chronicles, #1)
3120                           Die for Me (Revenants, #1)
48                                New Moon (Twilight, #2)
1781              The Dream Thieves (The Raven Cycle, #2)
542                  Last Sacrifice (Vampire Academy, #6)
7287    The Lost Prince (The Iron Fey: Call of the For...
6386        The Forbidden Game (The Forbidden Game, #1-3)
2996                     The Fiery Heart (Bloodlines, #4)
7775                                 Destined (Wings, #4)
Name: title, dtype: object

En este caso, la similaridad devuelve títulos de la saga, pero no solo eso, sino que incluye algún otro título con la misma temática. Esta diferencia también se plasmaría en el sistema de recomendación.

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Salvar la matriz `data_books_ld` en el archivo `./data/data_books_lda.pickle`.

In [66]:
import pickle

with open("./data/data_books_lda.pickle", "bw") as f:
    f.write(pickle.dumps(data_books_lda))# Completar

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>


---

<a id="Anexo"></a> 
##  <font color="#00586D"> Anexo. Obtención de los resúmenes mediante scraping   </font>
<br>

<br>
Goodreads proporciona una API. Sin embargo, debido a que esta es algo obsoleta, y a que solamente se necesita la información relativa a los resúmenes, es posible obtenerla mediante scraping.

<div class="alert alert-block alert-danger">
<i class="fa fa-exclamation-triangle" aria-hidden="true"></i>
En sitios como IMDB está prohibido. Aquí no pone nada. Sin embargo, el sitio se proteje del exceso de peticiones, y no deja hacer más de una cada diez segundos ya que, cuando detecta que se supera este umbral, devuelve documentos aleatoriamente. 
</div>

Las páginas tienen el formato `https://www.goodreads.com/book/show/book_id`. Por ejemplo,  [https://www.goodreads.com/book/show/320](https://www.goodreads.com/book/show/320) accede a la pagina relativa al libro "100 años de soledad" de Gabriel García Márquez. Al inspeccionar la página se encuentra que un objeto denominado `readable stacked` que contiene el resumen (a veces una versión larga y una corta). Es necesario extraer la última. 

In [67]:
import requests
from bs4 import BeautifulSoup

def scrap_book_overview(book_id, save=False):
    try:
        # Se conecta a la página
        url = "https://www.goodreads.com/book/show/"+str(book_id)
        response = requests.get(url)
        # Crea el objeto beautiful soup
        soup = BeautifulSoup(response.text, 'lxml')
        # Accede al componente
        sec = soup.find("div", {"class": "readable stacked"})
        # Extrae la revisión
        overview = sec.findAll('span')[-1]
        # La almacena si procede
        if not overview.text is None and save:
            file = open("overviews/"+str(book_id)+".txt","w") 
            file.write(overview.text)
            file.close()
        return overview.text
    except:
        return None
    
scrap_book_overview(320)

'The brilliant, bestselling, landmark novel that tells the story of the Buendia family, and chronicles the irreconcilable conflict between the desire for solitude and the need for love—in rich, imaginative prose that has come to define an entire genre known as "magical realism."'

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#00586D"></i> </font></div>